In [116]:
import numpy as np
import seaborn as sns
import pandas as pd

In [117]:
summary_df = pd.read_csv('data/Munroe-Streetcar-Info/Streetcar_data-tabulated.csv')
summary_df.head()

,"Veh type (LFsc = 0, Bus = 1)",No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,Summary date (mm-dd-yyyy),Date of Last Change (mm-dd-yyyy),RT dist (km),First dep NB or WB,First dep SB or EB,Last dep NB or WB,Last dep SB or EB,Route,Interruption
0,0,16,10:00,152,8,12.8,0,0,02-08-2026 : 03-14-2026,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
1,0,17,10:00,164,6,11.9,1,0,02-08-2026 : 03-14-2026,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
2,0,19,10:00,182,8,10.7,2,0,02-08-2026 : 03-14-2026,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
3,0,16,10:00,148,12,13.2,3,0,02-08-2026 : 03-14-2026,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
4,0,14,10:00,132,8,14.8,4,0,02-08-2026 : 03-14-2026,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN


In [118]:
# We drop the car rows and the summary date.
columns = summary_df.columns
Veh = [x for x in columns if 'Veh type' in x][0]
summary_date = [x for x in columns if 'Summary' in x][0]
summary_df = summary_df[summary_df[Veh] == 0]
summary_df = summary_df.drop(columns=[Veh, summary_date])

#update the columns
columns = summary_df.columns
summary_df.head()


,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,Date of Last Change (mm-dd-yyyy),RT dist (km),First dep NB or WB,First dep SB or EB,Last dep NB or WB,Last dep SB or EB,Route,Interruption
0,16,10:00,152,8,12.8,0,0,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
1,17,10:00,164,6,11.9,1,0,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
2,19,10:00,182,8,10.7,2,0,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
3,16,10:00,148,12,13.2,3,0,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN
4,14,10:00,132,8,14.8,4,0,12-22-2025,32.49,04:39,05:06,01:11,01:09,506 High Park-Main St Stn,NaN


In [119]:
# My column names are kind of awful so I set constants to address this
DOLC = [x for x in columns if 'Last Change' in x][0]
WEEK = [x for x in columns if 'Week' in x][0]
TIME = [x for x in columns if 'Time' in x][0]

In [120]:
hourly_df = pd.DataFrame()
# number of days from 01-01-2023 to 02-28-2026
import datetime as dt
feb_28_2026 = dt.datetime(year=2026, month=2, day=28)
jan_14_2023 = dt.datetime(year=2023, month=1, day=14)
print((feb_28_2026 - jan_14_2023).days)

1141


In [121]:
if summary_df[DOLC].dtype != np.datetime64:
    summary_df[DOLC] = summary_df[DOLC].apply(lambda x: pd.to_datetime(x, format='%m-%d-%Y'))

In [122]:
daily_df = dict()
for s in range(5): #time of the week
    daily_df[s] = pd.DataFrame()
    daily_df[s]['date'] = pd.date_range(start=jan_14_2023, end=feb_28_2026, freq='D')

for s in range(5):
    rows = daily_df[s].shape[0]
    for column in columns:
        # Next we update the rows based on each date in DoLC_list[k]
        list_to_fill = []
        for row in range(rows):
            date=daily_df[s]['date'].iloc[row]
            day_of_week = date.dayofweek

            if day_of_week == 5:
                k=1
            elif day_of_week == 6:
                k=2
            else:
                k=0

            # Picks out the first Date of Last Change before the current date.
            date_last_changed = sorted(summary_df[(summary_df[TIME] == s) & (summary_df[WEEK] == k) & (summary_df[DOLC] <= date)][DOLC].values)[-1]

            list_to_fill.append(summary_df[(summary_df[TIME] == s) & (summary_df[WEEK] == k) & (summary_df[DOLC] == date_last_changed)][column].iloc[0])

        daily_df[s][column] = list_to_fill




In [123]:
daily_df = pd.concat(list(daily_df.values()))

In [124]:
daily_df.sort_values(['date', TIME], inplace=True)
display(daily_df[:20])

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),Time of Day (morning: 0-late evening: 4)*,Weekday=0/Sat=1/Sun=2,Date of Last Change (mm-dd-yyyy),RT dist (km),First dep NB or WB,First dep SB or EB,Last dep NB or WB,Last dep SB or EB,Route,Interruption
0,2023-01-14,12,10:00,106,14,18.4,0,1,2023-01-14,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN
0,2023-01-14,16,09:30,138,14,14.2,1,1,2023-01-14,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN
0,2023-01-14,20,08:30,156,14,12.5,2,1,2023-01-14,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN
0,2023-01-14,15,10:00,134,16,14.6,3,1,2023-01-14,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN
0,2023-01-14,14,10:00,130,10,15.0,4,1,2023-01-14,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN
1,2023-01-15,11,10:00,102,8,19.2,0,2,2023-01-08,32.56,05:15,06:00,01:18,00:45,506 High Park-Main Street Stn,NaN
1,2023-01-15,14,10:00,130,10,15.0,1,2,2023-01-08,32.56,05:15,06:00,01:18,00:45,506 High Park-Main Street Stn,NaN
1,2023-01-15,18,09:00,148,14,13.2,2,2,2023-01-08,32.56,05:15,06:00,01:18,00:45,506 High Park-Main Street Stn,NaN
1,2023-01-15,14,09:30,122,11,16.0,3,2,2023-01-08,32.56,05:15,06:00,01:18,00:45,506 High Park-Main Street Stn,NaN
1,2023-01-15,13,09:30,114,9,17.1,4,2,2023-01-08,32.56,05:15,06:00,01:18,00:45,506 High Park-Main Street Stn,NaN


In [125]:
# Finally, let's remove the TIME and WEEK columns by giving specific timeframes.
# First let's write a dictionary to translate a (WEEK, TIME) pair to the appropriate
# range of time. Note that saturdays have the same schedule as sundays, so we only mark the saturday
# key.

# NOTE: because services seem to begin *before/end* the assigned time periods, we shift to '4 am' at the beginning of a day
# A more accurate timeframe would use the first/last departure times, which we can make if we need to.
translator = {
    (0,0): (4,9),
    (0,1): (9,15),
    (0,2): (15,19),
    (0,3): (19,22),
    (0,4): (22,28), #28 denotes the fact that it's 4 am on the next day
    (1,0): (4,8),
    (1,1): (8,12),
    (1,2): (12,19),
    (1,3): (19,22),
    (1,4): (22,28)
}


rows = daily_df.shape[0]
time_period_start = []
time_period_end = []
for row in range(rows):
    time = daily_df[TIME].iloc[row]
    week = int(daily_df[WEEK].iloc[row] != 0) #equal to 1 if the WEEK row is not zero
    current_date = daily_df['date'].iloc[row]

    hour_start = translator[(week,time)][0]
    hour_end = translator[(week,time)][1]

    start_date = pd.Timestamp(year=current_date.year, month=current_date.month, day=current_date.day, hour=hour_start)
    # For the end date, we need to account for the time being +1 day.
    if hour_end > 24:
        end_date = pd.Timestamp(year=current_date.year, month=current_date.month, day=current_date.day, hour=hour_end//24) + pd.Timedelta(days=1)
    else:
        end_date = pd.Timestamp(year=current_date.year, month=current_date.month, day=current_date.day, hour=hour_end)
    time_period_start.append(start_date)
    time_period_end.append(end_date)
daily_df['time period start'] = time_period_start
daily_df['time period end'] = time_period_end

# Finally, drop the TIME, WEEK, and DOLC columns:
daily_df.drop(columns=[TIME, WEEK, DOLC], inplace=True)

In [126]:
daily_df.head()

,date,No. of Veh,Service interval,Run time (min),Term time (min),Avg. spd (km/h),RT dist (km),First dep NB or WB,First dep SB or EB,Last dep NB or WB,Last dep SB or EB,Route,Interruption,time period start,time period end
0,2023-01-14,12,10:00,106,14,18.4,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN,2023-01-14 04:00:00,2023-01-14 08:00:00
0,2023-01-14,16,09:30,138,14,14.2,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN,2023-01-14 08:00:00,2023-01-14 12:00:00
0,2023-01-14,20,08:30,156,14,12.5,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN,2023-01-14 12:00:00,2023-01-14 19:00:00
0,2023-01-14,15,10:00,134,16,14.6,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN,2023-01-14 19:00:00,2023-01-14 22:00:00
0,2023-01-14,14,10:00,130,10,15.0,32.56,05:15,06:00,01:46,01:05,506 High Park-Main Street Stn,NaN,2023-01-14 22:00:00,2023-01-15 01:00:00


In [127]:
daily_df.to_csv('data/Munroe-Streetcar-Info/Streetcar_data_cleaned_up.csv')